# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess

if not os.path.isdir("Flyranks-internship-assignmnet-1"):
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/MaryamNaveed-bioinfo/Flyranks-internship-assignmnet-1"], check=True)
os.chdir("Flyranks-internship-assignmnet-1")
print("Working dir:", os.getcwd())

# Install duckdb if needed
subprocess.run(["pip", "install", "-q", "duckdb"], check=True)

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query:", REL)

Working dir: /content/Flyranks-internship-assignmnet-1
Connected. Ready to query: hf://datasets/FlyRank/internship-warehouse


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content page, on one specific day
(client x content x day), from `fact_content_daily_performance`.

**Time window:** I'll use a mid-panel month `month=2026-03` as recommended,
since the most recent month is reserved as a sealed test window and shouldn't be
used to build or tune label logic.

In [2]:
q = f"""
SELECT *
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
"""
preview = con.sql(q).df()
preview

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
q = f"""
SELECT *
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 0
"""
schema = con.sql(q).df()
print(schema.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


**Feature** (inputs the model can use):
- `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position` search visibility
  and ranking performance signals
- `ga4_pageviews`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`
  on-page engagement signals
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`
  traffic-source mix, useful context for understanding where visibility comes from
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` availability
  flags, needed to filter out rows where a metric simply wasn't being tracked yet

**Label / proxy** (what I'd predict or rank):
- A future decline/opportunity signal built from `gsc_impressions` and `gsc_clicks` trends
  comparing a prior feature window against a later target window. The exact threshold and
  window definition will be built in a later week; this week only establishes which raw
  columns it would be built from.

**Context** (background info, not fed directly into a model):
- `client_hash_id`, `content_hash_id` join keys, used for grouping/joining only
- `report_date`, `month` — used to define time windows, not as raw model inputs

**Excluded** (deliberately not using):
- `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`,
  `ai_meta`, `ai_other` — excluded because AI-referral data is extremely sparse
  (~30K rows out of ~79M total), making it unreliable as a feature or label this early.
- Any FlyRank internal product score (e.g. `health_score`, `priority_score`) not present
  in this data at all, and would be excluded even if it were, to avoid the model just
  learning to copy FlyRank's existing rule instead of finding real signal.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 : Grain check:** confirm there are no duplicate rows for the same
client x content x day combination (proving one row truly is one observation).

In [4]:
q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_combos
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q1).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_combos
0,9841378,9841378


**Query 2 : Row count and date span:** confirm how many rows this slice
contains, and which dates it actually covers.

In [5]:
q2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q2).df()

,row_count,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


**Query 3 : Availability check:** filter to rows where GSC data is actually
available, and see how many survive versus the full slice.

In [6]:
q3_total = f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
q3_available = f"""
SELECT COUNT(*) AS gsc_available_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
"""
total = con.sql(q3_total).df()
available = con.sql(q3_available).df()
print(total)
print(available)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows
0     9841378
   gsc_available_rows
0             3611061


**Results:**
- **Grain confirmed:** total_rows (9,841,378) exactly equals distinct_combos (9,841,378)
  no duplicates. One row genuinely is one client x content x day observation, as claimed.
- **Row count and date span confirmed:** this slice contains 9,841,378 rows, spanning
  2026-03-01 through 2026-03-31 the full month, as intended.
- **Availability check:** only 3,611,061 of 9,841,378 rows (about 36.7%) have
  `gsc_data_available IS TRUE`. This means nearly two-thirds of rows in this month lack
  usable Search Console data a significant availability gap that any feature or label
  built from GSC columns must account for by filtering first.

**Five features, each "knowable at the decision moment because…":**

1. `gsc_impressions` knowable at the decision moment because it's a direct measurement
   of search visibility already logged for that day, before any future outcome occurs.
2. `gsc_avg_position` knowable because it reflects where the page currently ranks,
   observed as of the report date, not a future value.
3. `ga4_engaged_sessions` knowable because it's a completed daily engagement count,
   already recorded by the time of the decision.
4. `sessions_organic`  knowable because it's an already-logged traffic count for that day.
5. `client_has_gsc` knowable because it's a static availability flag, known before any data is even queried.

In [7]:
features_q = f"""
SELECT
    client_hash_id, content_hash_id, report_date,
    gsc_impressions, gsc_avg_position, ga4_engaged_sessions,
    sessions_organic, client_has_gsc
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
LIMIT 20
"""
feature_frame = con.sql(features_q).df()
feature_frame

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,sessions_organic,client_has_gsc
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,3.350000,<NA>,<NA>,True
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0.000000,<NA>,<NA>,True
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,4.928000,<NA>,<NA>,True
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,4.000000,<NA>,<NA>,True
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,2.272727,<NA>,<NA>,True
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,7.347280,<NA>,<NA>,True
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,7.832461,<NA>,<NA>,True
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,3.272727,<NA>,<NA>,True
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,5.636364,<NA>,<NA>,True
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,4.500000,<NA>,<NA>,True


**The leakage trap:** I'll build a quick score using legitimate features, then deliberately
add a label-derived column and watch the score jump toward perfect  proving I understand
what leakage looks like before I avoid it in real modeling.

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sample_q = f"""
SELECT gsc_impressions, gsc_clicks, gsc_avg_position, gsc_sum_position
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
LIMIT 50000
"""
sample = con.sql(sample_q).df()

sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"]
sample["high_ctr_label"] = (sample["ctr"] > 0.05).astype(int)

X_honest = sample[["gsc_impressions", "gsc_avg_position"]].fillna(0)
y = sample["high_ctr_label"]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"HONEST score (no leakage): {honest_score:.3f}")

X_leaky = sample[["gsc_impressions", "gsc_avg_position", "gsc_clicks"]].fillna(0)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train2, y_train2)
leaky_score = roc_auc_score(y_test2, model2.predict_proba(X_test2)[:, 1])
print(f"LEAKY score (gsc_clicks included): {leaky_score:.3f}")

print(f"\nScore jumped by {leaky_score - honest_score:.3f} just by adding the column the label was derived from.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST score (no leakage): 0.694
LEAKY score (gsc_clicks included): 1.000

Score jumped by 0.306 just by adding the column the label was derived from.


**Result : the trap:**

The honest model (using only `gsc_impressions` and `gsc_avg_position`) scored 0.694
a believable, imperfect result for a real prediction task.

The leaky model  with `gsc_clicks` added — scored a suspicious 1.000. This is not a
genuine improvement: `gsc_clicks` was directly used to calculate the label itself
(`ctr = gsc_clicks / gsc_impressions`, then thresholded into `high_ctr_label`), so the
model wasn't predicting anything  it was just reading the answer back out of a
disguised copy of itself.

**I am removing `gsc_clicks` from the feature set going forward** and keeping the honest
0.694 score as the real, trustworthy baseline. This confirms the same leakage lesson from
the starter notebook, now demonstrated on real warehouse data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has real limits worth naming honestly:

- **Uneven availability:** only ~36.7% of rows in this month have GSC data actually
  available — the remaining rows would need to be filtered out or handled separately,
  not treated as "zero traffic."
- **Unbalanced client history:** different clients started tracking at different times
  (`gsc_data_start`, `ga4_data_start` in `dim_clients`), so early rows for newer clients
  may only have partial signal, not a true reflection of performance.
- **GSC-only early rows:** some rows may have search data but no GA4 engagement data yet,
  meaning engagement-based features can't be computed for every row equally.
- **No causal information:** this data can show correlations and patterns, but cannot
  prove that any specific action (like a content refresh) caused a change in outcome.
- **Window overlap risk:** since I haven't yet built time-based labels, care will be needed
  later to ensure feature windows and target windows never overlap, to avoid leakage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.